# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [104]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [105]:
# TODO: Import the necessary libs
# For example: 
import os
import json
from lib.llm import LLM
from lib.messages import ToolMessage, AIMessage, BaseMessage, SystemMessage, UserMessage
from lib.tooling import tool
from dotenv import load_dotenv

import chromadb
from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection
from typing import TypedDict, List

from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource

from tavily import TavilyClient

In [106]:
# TODO: Load environment variables
load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

True

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [107]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game 
@tool
def retrieve_game(query):
    print("[LOG] Retrieve Game tool")
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embeddings_fn
    )
    results = collection.query(
        query_texts=[query],
        include=["documents"]
    )
    return results["documents"][0]

# NOTE: useful resource -> https://docs.trychroma.com/docs/querying-collections/query-and-get

#### Evaluate Retrieval Tool

In [108]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result  
@tool 
def evaluate_retrieval(question, retrieved_docs):
    print("[LOG] Evaluate Retrieval tool")
    llm = LLM(
        model="gpt-4o-mini",
        temperature=0.3,
    )
    msg = [
        SystemMessage(content=(
            "Evaluate whether the retrieved documents contain enough information to answer "
            "the user's question at the level of detail available in the dataset. "
            "A release year is sufficient for a question asking when a game was released; "
            "an exact day and month are not required. "
            "If at least one document contains the requested game and the relevant year, "
            "platform, publisher, or description, set useful to true. "
            "Ignore unrelated retrieved documents. "
            "Return only JSON in this exact form: "
            '{"useful": true, "description": "reason"}. '
            "Set useful to false only if none of the documents contain enough information "
            "to answer the question."
        )),
        UserMessage(content=(
            f"Question {question}"
            f"Retrieved documents: {retrieved_docs}"
        ))

    ]

    result = llm.invoke(msg).content
    # print(result)
    return result
    

#### Game Web Search Tool

In [109]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
#--
# NOTE: almost verbatim from M06_solution_exercise

from typing import Dict

@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """

    # Sprint("[LOG] Web search tool")
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
        max_results=3
    )
    
    web_documents = []

    if search_result.get("answer"):
        web_documents.append(search_result["answer"])

    for result in search_result.get("results", []):
        web_documents.append(
            f"Title: {result.get('title', '')}\n"
            f"Content: {result.get('content', '')}\n"
            f"Source: {result.get('url', '')}"
        )

    return {
        "documents": web_documents,
        "source": "web_search"
    }

### Agent

NOTE: ALL the next cells were copied from M08_demo.ipynb

In [110]:
class State(TypedDict, total=False):
    messages: List[BaseMessage]
    question: str
    documents: List[str]
    answer: str
    reasoning: str
    tool_usage: List[str]

In [111]:
def augment(state:State):
    question = state["question"]
    documents = state["documents"]
    context = "\n\n".join(documents)

    messages = [
        SystemMessage(content="You are an assistant for question-answering tasks."),
        UserMessage(
            content=(
                "Answer the question using the retrieved context. "
                "First call evaluate_retrieval with the original question and retrieved context. "
                "If useful=true, answer using the retrieved context. "
                "If useful=false, call game_web_search with the original question. "
                "When using web-search results, include the provided Source URL as a citation "
                "in the final answer. Never call game_web_search before evaluate_retrieval."
                f"\n\nQuestion:\n{question}"
                f"\n\nRetrieved context:\n{context}"
            )
        )
    ]

    return {"messages": messages}

In [112]:
def generate(state: State, resource: Resource): 
    llm: LLM = resource.vars.get("llm") 
    messages = list(state["messages"]) 
    available_tools = { 
        "evaluate_retrieval": evaluate_retrieval, 
        "game_web_search": game_web_search
    }

    reasoning = ""
    tool_usage = []
    
    while True: 
        ai_message = llm.invoke(messages) 
        clean_ai_message = AIMessage(
            role="assistant",
            content=ai_message.content,
            tool_calls=ai_message.tool_calls
        )

        messages.append(clean_ai_message)
        
        if not ai_message.tool_calls: 
            return { 
                "answer": ai_message.content, 
                "messages": messages,
                "reasoning": reasoning,
                "tool_usage": tool_usage
            } 
            
        for tool_call in ai_message.tool_calls: 
            tool_name = tool_call.function.name
            tool_usage.append(tool_name)

            tool_arguments = json.loads(tool_call.function.arguments) 
            selected_tool = available_tools[tool_name] 
            tool_result = selected_tool.func(**tool_arguments)

            if tool_name == "evaluate_retrieval":
                reasoning = str(tool_result)

            messages.append( 
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call.id, 
                    name=tool_name
                ) 
            )

In [113]:
def retrieve(state:State, resource:Resource):
    question = state["question"]
    collection:Collection = resource.vars.get("collection")

    # get the data from the database
    results = collection.query(
        query_texts=[question],
        n_results=3,
        include=['documents']
    )
    retrieved_docs = results['documents'][0]
    
    return {
        "documents": retrieved_docs,
        "source": "knowledge_base"
    }

In [114]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embeddings_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embeddings_fn
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [115]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
# --
# NOTE: Copied almost verbatim from M08_demo.ipynb
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource

workflow = StateMachine(State)

# Create steps
entry = EntryPoint()
retrieve_step = Step("retrieve", retrieve)
augment_step = Step("augment", augment)
generate_step = Step("generate", generate)
termination = Termination()
        
workflow.add_steps(
    [
        entry, 
        retrieve_step,
        augment_step, 
        generate_step, 
        termination
    ]
)

# Add transitions
workflow.connect(entry, retrieve_step)
workflow.connect(retrieve_step, augment_step)
workflow.connect(augment_step, generate_step)
workflow.connect(generate_step, termination)

llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
    #tools=[game_web_search, evaluate_retrieval, retrieve_game]
    tools=[evaluate_retrieval, game_web_search]
)

resource = Resource(
    vars={
        "llm": llm,
        "collection": collection
    }
)

In [116]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
user_prompts = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]

In [117]:
def invoke_agent(query):
    initial_state = {
        "question": query
    }

    run_object = workflow.run(initial_state, resource)
    final_state = run_object.get_final_state()

    print(f"\nQUERY:\n{query}")
    print(f"\nREASONING:\n{final_state['reasoning']}")
    print(f"\nTOOLS USED:\n{', '.join(final_state['tool_usage'])}")
    print(f"\nFINAL ANSWER:\n{final_state['answer']}")

In [118]:
invoke_agent(user_prompts[0])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[LOG] Evaluate Retrieval tool
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

QUERY:
When Pokémon Gold and Silver was released?

REASONING:
{"useful": true, "description": "The document contains the release year (1999) for Pokémon Gold and Silver."}

TOOLS USED:
evaluate_retrieval

FINAL ANSWER:
Pokémon Gold and Silver was released in 1999.


In [119]:
invoke_agent(user_prompts[1])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[LOG] Evaluate Retrieval tool
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

QUERY:
Which one was the first 3D platformer Mario game?

REASONING:
{"useful": true, "description": "The document mentions Super Mario 64, which is identified as the first 3D platformer Mario game released in 1996."}

TOOLS USED:
evaluate_retrieval

FINAL ANSWER:
The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996. It was a groundbreaking title that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.


In [120]:
invoke_agent(user_prompts[2])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[LOG] Evaluate Retrieval tool
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

QUERY:
Was Mortal Kombat X realeased for Playstation 5?

REASONING:
{"useful": false, "description": "No relevant information about Mortal Kombat X or its release on PlayStation 5 is found in the retrieved documents."}

TOOLS USED:
evaluate_retrieval, game_web_search

FINAL ANSWER:
Mortal Kombat X was released for the PlayStation 5 as part of the PlayStation Plus Collection. The digital edition became available on October 30, 2025, and it was included among the retro titles for PS Plus subscribers on the next-gen console. 

For more details, you can check the source [here](https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V).


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes